# 🎙️ TTT-Dialect: AI Hub 노인·방언 음성 인식

**시작 전 필수:** 런타임 → 런타임 유형 변경 → **T4 GPU** 선택

---
**사용 데이터**
- 명령어 음성(노인남녀) — AI Hub
- 중·노년층 한국어 방언 데이터 (강원도, 경상도) — AI Hub

## Step 1 — Drive 연동 및 경로 확인

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os

# Drive 전체에서 zip 파일 위치 자동 탐색
print('Drive에서 AI Hub 데이터 탐색 중...')
zip_files = []
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for f in files:
        if f.endswith('.zip'):
            full_path = os.path.join(root, f)
            zip_files.append(full_path)
            print(f'  발견: {full_path}')

print(f'\n총 zip 파일: {len(zip_files)}개')

## Step 2 — zip 압축 해제

In [ ]:
import zipfile, os

ELDERLY_OUT = '/content/data/elderly'
DIALECT_OUT = '/content/data/dialect'
os.makedirs(ELDERLY_OUT, exist_ok=True)
os.makedirs(DIALECT_OUT, exist_ok=True)

# zip당 최대 추출 개수 (디스크 절약)
MAX_PER_ZIP = 500

def extract_selective(zip_path, out_dir, max_count=500):
    """wav + 대응 json만 max_count개 선택 추출"""
    with zipfile.ZipFile(zip_path, 'r') as z:
        members = z.namelist()
        wav_members = [m for m in members if m.endswith('.wav')]
        print(f'  전체 wav: {len(wav_members)}개 → {min(max_count, len(wav_members))}개만 추출')

        selected_wav = wav_members[:max_count]
        to_extract = set()
        for wav in selected_wav:
            to_extract.add(wav)
            # 같은 이름의 json도 함께 추출
            json_name = wav.replace('.wav', '.json')
            if json_name in members:
                to_extract.add(json_name)

        for member in to_extract:
            try:
                z.extract(member, out_dir)
            except:
                pass

    return len(selected_wav)

elderly_wavs, dialect_wavs = [], []

for zip_path in zip_files:
    fname = os.path.basename(zip_path)

    if any(kw in fname for kw in ['명령어', '키오스크', '노인', '노년']):
        out = ELDERLY_OUT
        tag = '노인'
    elif any(kw in fname for kw in ['경상', '강원', '전라', '충청', '제주', '방언']):
        out = DIALECT_OUT
        tag = '방언'
    else:
        out = ELDERLY_OUT
        tag = '기타'

    print(f'[{tag}] 선택 추출 중: {fname}')
    try:
        n = extract_selective(zip_path, out, MAX_PER_ZIP)
        print(f'  ✅ {n}개 추출 완료')
    except Exception as e:
        print(f'  ❌ 오류: {e}')

# 결과 수집
for r, d, files in os.walk(ELDERLY_OUT):
    for f in files:
        if f.endswith('.wav'):
            elderly_wavs.append(os.path.join(r, f))

for r, d, files in os.walk(DIALECT_OUT):
    for f in files:
        if f.endswith('.wav'):
            dialect_wavs.append(os.path.join(r, f))

# 디스크 사용량 확인
import shutil
disk = shutil.disk_usage('/')
print(f'\n✅ 추출 완료!')
print(f'   노인 음성: {len(elderly_wavs)}개')
print(f'   방언:      {len(dialect_wavs)}개')
print(f'   디스크 남은 용량: {disk.free / 1e9:.1f} GB')

## Step 3 — 패키지 설치 및 코드 클론

In [ ]:
import os

REPO_DIR = '/content/TTT-Dialect'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/kts6450/TTT-Dialect.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

!pip install -q openai-whisper transformers librosa soundfile jiwer loguru pyyaml accelerate torchaudio
print('✅ 완료!')

## Step 4 — GPU 및 모델 로드

In [ ]:
import sys, torch
sys.path.insert(0, '/content/TTT-Dialect')

print(f'GPU: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'모델: {torch.cuda.get_device_name(0)}')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

from models.base_whisper import KoreanWhisperModel

MODEL_NAME = 'SungBeom/whisper-small-ko'
print(f'\n모델 로드 중: {MODEL_NAME}')
model = KoreanWhisperModel(MODEL_NAME)
model.model = model.model.to(DEVICE)

params = model.count_parameters()
print(f'✅ 모델 로드 완료! 파라미터: {params["total"]:,}')

## Step 5 — AI Hub 데이터 manifest 생성

In [ ]:
import json, librosa, os
from tqdm.notebook import tqdm

SR = 16000
manifest = []

def parse_aihub_label(json_path):
    """AI Hub 공통 라벨 파싱 (전사정보.LabelText 구조)"""
    with open(json_path, encoding='utf-8') as jf:
        label = json.load(jf)

    transcript = label.get('전사정보', {}).get('LabelText', '').strip()
    speaker    = label.get('화자정보', {})
    dialect    = speaker.get('Dialect', '서울').strip()
    age_str    = speaker.get('Age', '60~69')
    # "60~69" → 65 (중간값)
    try:
        age = int(age_str.split('~')[0])
    except:
        age = 65

    return transcript, dialect, age

# ── 노인 음성 처리 ─────────────────────────────────────
print(f'노인 음성 처리 중: {len(elderly_wavs)}개')
skip = 0
for wav_path in tqdm(elderly_wavs[:1000]):
    try:
        json_path = wav_path.replace('.wav', '.json')
        if not os.path.exists(json_path):
            skip += 1
            continue

        transcript, dialect, age = parse_aihub_label(json_path)
        if not transcript:
            skip += 1
            continue

        audio = librosa.load(wav_path, sr=SR)[0]
        duration = len(audio) / SR
        if duration < 0.3 or duration > 30:
            continue

        manifest.append({
            'audio_path': wav_path,
            'transcript': transcript,
            'dialect': dialect,
            'speaker_age': age,
            'speaker_id': os.path.basename(wav_path)[:12],
            'duration_sec': round(duration, 2),
            'type': 'elderly'
        })
    except Exception as e:
        skip += 1

print(f'  → 추가: {sum(1 for m in manifest if m["type"]=="elderly")}개 / 스킵: {skip}개')

# ── 방언 데이터 처리 ───────────────────────────────────
print(f'\n방언 데이터 처리 중: {len(dialect_wavs)}개')
skip = 0
for wav_path in tqdm(dialect_wavs[:1000]):
    try:
        json_path = wav_path.replace('.wav', '.json')
        if not os.path.exists(json_path):
            skip += 1
            continue

        transcript, dialect, age = parse_aihub_label(json_path)
        if not transcript:
            skip += 1
            continue

        audio = librosa.load(wav_path, sr=SR)[0]
        duration = len(audio) / SR
        if duration < 0.3 or duration > 30:
            continue

        manifest.append({
            'audio_path': wav_path,
            'transcript': transcript,
            'dialect': dialect,
            'speaker_age': age,
            'speaker_id': os.path.basename(wav_path)[:12],
            'duration_sec': round(duration, 2),
            'type': 'dialect'
        })
    except Exception as e:
        skip += 1

print(f'  → 추가: {sum(1 for m in manifest if m["type"]=="dialect")}개 / 스킵: {skip}개')

# manifest 저장
MANIFEST_PATH = '/content/manifest.jsonl'
with open(MANIFEST_PATH, 'w', encoding='utf-8') as f:
    for m in manifest:
        f.write(json.dumps(m, ensure_ascii=False) + '\n')

print(f'\n✅ manifest 생성 완료!')
print(f'   전체: {len(manifest)}개')
print(f'   노인: {sum(1 for m in manifest if m["type"]=="elderly")}개')
print(f'   방언: {sum(1 for m in manifest if m["type"]=="dialect")}개')
if manifest:
    print(f'\n예시 전사: "{manifest[0]["transcript"]}"')
    print(f'방언: {manifest[0]["dialect"]} / 나이: {manifest[0]["speaker_age"]}')

## Step 6 — 라벨 구조 확인 (manifest가 0개일 때 실행)

In [ ]:
# json 구조 전체 확인 — 반드시 실행하세요
import json, os

found = 0
for root, dirs, files in os.walk('/content/data'):
    for f in sorted(files):
        if f.endswith('.json'):
            json_path = os.path.join(root, f)
            print(f'\n=== {json_path} ===')
            try:
                with open(json_path, encoding='utf-8') as jf:
                    data = json.load(jf)
                print('최상위 키:', list(data.keys()))
                # 중첩 구조 펼쳐서 출력
                print(json.dumps(data, ensure_ascii=False, indent=2)[:800])
            except Exception as e:
                print(f'오류: {e}')
            found += 1
            if found >= 3:   # 3개만 보면 충분
                break
    if found >= 3:
        break

if found == 0:
    print('json 파일이 없습니다. wav 파일 위치 확인:')
    for root, dirs, files in os.walk('/content/data'):
        for f in files[:5]:
            print(' ', os.path.join(root, f))

## Step 7 — TTT 어댑터 초기화

In [ ]:
from models.ttt_adapter import TTTAdapter

adapter = TTTAdapter(
    base_model=model,
    profile_dir='/content/user_profiles',
    top_k_layers=2,
    lr=1e-4,
    adaptation_steps=30,
)
print('✅ TTT 어댑터 초기화 완료!')

## Step 8 — TTT 이전 WER 측정

In [ ]:
import json, librosa, torch
from jiwer import wer
from tqdm.notebook import tqdm

SR = 16000
N_TEST = 50

samples = []
with open(MANIFEST_PATH) as f:
    for line in f:
        samples.append(json.loads(line))

print(f'전체 샘플: {len(samples)}개')

if len(samples) == 0:
    print('\n❌ manifest가 비어있습니다!')
    print('→ Step 6 셀을 실행해서 json 키 구조를 확인하세요.')
    print('→ 확인 후 Step 5 셀의 transcript 파싱 부분을 수정해야 합니다.')
    raise SystemExit('manifest 비어있음 — Step 6 실행 필요')

test_samples = samples[:N_TEST]

def infer(sample, mdl):
    audio, _ = librosa.load(sample['audio_path'], sr=SR)
    feat = mdl.processor.feature_extractor(
        audio, sampling_rate=SR, return_tensors='pt'
    ).input_features.to(DEVICE)
    with torch.no_grad():
        result = mdl.transcribe(feat)
    return result[0] if result else ''

print('TTT 이전 WER 측정 중...')
refs, hyps = [], []
for s in tqdm(test_samples):
    refs.append(s['transcript'])
    hyps.append(infer(s, model))

wer_before = wer(refs, hyps)
print(f'\n📊 TTT 이전 WER: {wer_before:.1%}')
if refs:
    print(f'   정답:  "{refs[0][:40]}"')
    print(f'   인식:  "{hyps[0][:40]}"')

## Step 9 — TTT 캘리브레이션 (핵심)

In [ ]:
N_CALIB = 20
calib_samples = samples[N_TEST:N_TEST + N_CALIB]

calib_features, calib_texts = [], []
for s in calib_samples:
    audio, _ = librosa.load(s['audio_path'], sr=SR)
    feat = model.processor.feature_extractor(
        audio, sampling_rate=SR, return_tensors='pt'
    ).input_features[0]
    calib_features.append(feat)
    calib_texts.append(s['transcript'])

print(f'TTT 캘리브레이션 시작: {N_CALIB}개 샘플 (약 1~2분)')

profile = adapter.calibrate(
    user_id='aihub_user_001',
    audio_features=calib_features,
    transcripts=calib_texts,
    dialect=calib_samples[0].get('dialect', '서울'),
    age=70,
)

print(f'\n✅ TTT 완료!')
print(f'   WER 이전: {profile.wer_before:.1%}')
print(f'   WER 이후: {profile.wer_after:.1%}')
print(f'   개선:     {profile.wer_before - profile.wer_after:.1%}p')

## Step 10 — 결과 시각화 및 Drive 저장

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('TTT 적용 전·후 성능 비교 (AI Hub 데이터)', fontsize=15, fontweight='bold')

# WER 비교
ax = axes[0]
labels = ['TTT 이전', 'TTT 이후']
values = [profile.wer_before * 100, profile.wer_after * 100]
colors = ['#FF6B6B', '#4ECDC4']
bars = ax.bar(labels, values, color=colors, alpha=0.85, width=0.5)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontweight='bold', fontsize=13)
ax.set_ylabel('WER (%) — 낮을수록 좋음')
ax.set_title('WER 비교')
ax.set_ylim(0, max(values) * 1.3)
ax.spines[['top', 'right']].set_visible(False)

# 적응 이력
ax2 = axes[1]
if profile.adaptation_history:
    steps = list(range(len(profile.adaptation_history)))
    wers = [h['wer'] * 100 for h in profile.adaptation_history]
    ax2.plot(steps, wers, 'o-', color='#667eea', linewidth=2.5, markersize=8)
    ax2.fill_between(steps, wers, alpha=0.15, color='#667eea')
    ax2.set_xlabel('적응 횟수')
    ax2.set_ylabel('WER (%)')
    ax2.set_title('적응 이력')
    ax2.spines[['top', 'right']].set_visible(False)

plt.tight_layout()

# Drive 저장
RESULT_DIR = '/content/drive/MyDrive/TTT-Dialect/results'
os.makedirs(RESULT_DIR, exist_ok=True)
plt.savefig(f'{RESULT_DIR}/wer_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# 요약 저장
summary = {
    'model': MODEL_NAME,
    'data': 'AI Hub 노인·방언',
    'n_test': N_TEST,
    'n_calib': N_CALIB,
    'wer_before': profile.wer_before,
    'wer_after': profile.wer_after,
    'improvement_pp': profile.wer_before - profile.wer_after,
    'improvement_pct': (profile.wer_before - profile.wer_after) / max(profile.wer_before, 1e-9) * 100
}
with open(f'{RESULT_DIR}/summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f'\n📊 최종 결과')
print(f'   TTT 이전 WER: {summary["wer_before"]:.1%}')
print(f'   TTT 이후 WER: {summary["wer_after"]:.1%}')
print(f'   개선량: {summary["improvement_pp"]:.1%}p ({summary["improvement_pct"]:.1f}%)')
print(f'\n💾 결과 저장 완료: {RESULT_DIR}')